# Diffusion Policy Push-T Training on Colab (Custom Implementation)

Train our custom U-Net 1D implementation on Push-T dataset.
Matches original repo architecture: conditional_unet1d + cosine schedule + EMA + DDIM.

**Checkpoints saved to Google Drive** - persists across sessions.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Setup checkpoint directory on Drive
import os
CKPT_DIR = '/content/drive/MyDrive/diffusion_policy_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CKPT_DIR}')

Mounted at /content/drive
Checkpoint dir: /content/drive/MyDrive/diffusion_policy_checkpoints


In [ ]:
# Check GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# Clone custom implementation from GitHub
!git clone https://github.com/VyDat-1702/Diffusion-Policy.git /content/diffusion
%cd /content/diffusion
!ls -la

Cloning into '/content/diffusion'...
remote: Enumerating objects: 887, done.
remote: Counting objects: 100% (887/887), done.
remote: Compressing objects: 100% (866/866), done.
remote: Total 887 (delta 20), reused 883 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (887/887), 30.25 MiB | 38.05 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/diffusion
total 88
drwxr-xr-x 11 root root 4096 Aug 22 06:37 .
drwxr-xr-x  1 root root 4096 Aug 22 06:37 ..
drwxr-xr-x  2 root root 4096 Aug 22 06:37 checkpoints
drwxr-xr-x  3 root root 4096 Aug 22 06:37 common
drwxr-xr-x  3 root root 4096 Aug 22 06:37 data
-rw-r--r--  1 root root 3798 Aug 22 06:37 diffusion_colab.ipynb
drwxr-xr-x  3 root root 4096 Aug 22 06:37 envs
-rw-r--r--  1 root root 3386 Aug 22 06:37 evaluate.py
drwxr-xr-x  8 root root 4096 Aug 22 06:37 .git
-rw-r--r--  1 root root  260 Aug 22 06:37 .gitignore
-rw-r--r--  1 root root 6140 Aug 22 06:37 infer_denoise.py
drwxr-xr-x  3 root root 4096 Aug 22 06:37 models
-rw-

In [ ]:
# Install dependencies
!pip install einops diffusers zarr pygame-ce pymunk -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 101.6 MB/s eta 0:00:00


In [ ]:
# Download Push-T dataset
!mkdir -p data/pusht
!wget -q https://diffusion-policy.cs.columbia.edu/data/training/pusht.zip -O /tmp/pusht.zip
!unzip -q /tmp/pusht.zip -d data/pusht/
!ls -la data/pusht/

total 16
drwxr-xr-x 4 root root 4096 Aug 22 06:38 .
drwxr-xr-x 3 root root 4096 Aug 22 06:37 ..
drwx------ 3 root root 4096 Feb 27  2023 pusht
drwxr-xr-x 4 root root 4096 Aug 22 06:37 pusht_cchi_v7_replay.zarr


In [ ]:
# Train custom implementation (500 epochs)
# Checkpoints saved to Google Drive automatically
%cd /content/diffusion
!python train_ddpm.py \
    --epochs 500 \
    --batch_size 256 \
    --device cuda \
    --use_ema \
    --lr_warmup_steps 500 \
    --zarr_path data/pusht/pusht_cchi_v7_replay.zarr \
    --ckpt_dir /content/drive/MyDrive/diffusion_policy_checkpoints

/content/diffusion
Model: U-Net 1D | Params: 65,353,218
Epoch 1/500: 100% 89/89 [00:24<00:00,  3.58it/s]
Epoch 1/500 | Loss: 0.849569 | LR: 1.78e-05
Epoch 2/500: 100% 89/89 [00:23<00:00,  3.76it/s]
Epoch 2/500 | Loss: 0.165103 | LR: 3.56e-05
Epoch 3/500: 100% 89/89 [00:24<00:00,  3.64it/s]
Epoch 3/500 | Loss: 0.077785 | LR: 5.34e-05
Epoch 4/500: 100% 89/89 [00:25<00:00,  3.53it/s]
Epoch 4/500 | Loss: 0.068310 | LR: 7.12e-05
Epoch 5/500: 100% 89/89 [00:24<00:00,  3.57it/s]
Epoch 5/500 | Loss: 0.062342 | LR: 8.90e-05
Epoch 6/500: 100% 89/89 [00:24<00:00,  3.60it/s]
Epoch 6/500 | Loss: 0.060120 | LR: 1.00e-04
Epoch 7/500: 100% 89/89 [00:24<00:00,  3.57it/s]
Epoch 7/500 | Loss: 0.055900 | LR: 1.00e-04
Epoch 8/500: 100% 89/89 [00:24<00:00,  3.57it/s]
Epoch 8/500 | Loss: 0.054237 | LR: 1.00e-04
Epoch 9/500: 100% 89/89 [00:24<00:00,  3.58it/s]
Epoch 9/500 | Loss: 0.051033 | LR: 1.00e-04
Epoch 10/500: 100% 89/89 [00:24<00:00,  3.58it/s]
Epoch 10/500 | Loss: 0.050834 | LR: 1.00e-04
Epoch 11/500

In [ ]:
# Evaluate after training (load from Google Drive)
%cd /content/diffusion
!python evaluate.py \
    --num_episodes 50 \
    --device cuda \
    --num_inference_steps 100 \
    --use_ddim True \
    --ckpt_path $(ls -t /content/drive/MyDrive/diffusion_policy_checkpoints/*.pt | head -1)

/content/diffusion
pygame-ce 2.5.8 (SDL 2.32.10, Python 3.13.15)
usage: evaluate.py [-h] [--num_episodes NUM_EPISODES] [--device DEVICE]
                   [--num_inference_steps NUM_INFERENCE_STEPS] [--use_ddim]
                   [--ckpt_path CKPT_PATH]
evaluate.py: error: unrecognized arguments: True


In [ ]:
# Visualize results (load from Google Drive)
%cd /content/diffusion
!python visualize.py --device cuda --ckpt_path $(ls -t /content/drive/MyDrive/diffusion_policy_checkpoints/*.pt | head -1)

/content/diffusion
Traceback (most recent call last):
  File "/content/diffusion/visualize.py", line 197, in <module>
    visualize_dataset_actions(save_path=os.path.join(PLOT_DIR, "dataset_actions.png"))
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/diffusion/visualize.py", line 88, in visualize_dataset_actions
    dataset = PushTReplayDataset(zarr_path, obs_horizon=2, action_horizon=8)
TypeError: PushTReplayDataset.__init__() got an unexpected keyword argument 'action_horizon'
